In [1]:
pip install ultralytics

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   -------- ------------------------------- 0.3/1.2 MB ? eta -:--:--
   -------- ------------------------------- 0.3/1.2 MB ? eta -:--:--
   -------- ------------------------------- 0.3/1.2 MB ? eta -:--:--
   ----------------- ---------------------- 0.5/1.2 MB 399.6 kB/s eta 0:00:02
   ----------------- ---------------------- 0.5/1.2 MB 399.6 kB/s eta 0:00:02
   ----------------- ---------------------- 0.5/1.2 MB 399.6 kB/s eta 0:00:02
   ----

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from ultralytics import YOLO

# โหลด pretrained model (เล็ก เหมาะกับ 4GB VRAM)
model = YOLO("yolov8n.pt")

# เริ่ม train
model.train(
    data="data.yaml",   # path ไปไฟล์ data.yaml
    epochs=50,
    imgsz=512,          # เพราะ dataset resize 512x512
    batch=8,            # ถ้า VRAM เต็ม ลดเป็น 4
    device=0,           # ใช้ GPU
    workers=4,
    project="runs",
    name="plate_train"
)

Ultralytics 8.4.14  Python-3.11.4 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=plate_train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, p

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,  40,  41,  42,  43,  44,  45,  46,  47,  49,  50,  51,  53,  55,  56,  57,  58,  59,  60,  61,  63,  64,  65,  66,
        68,  69,  70,  73,  76,  79,  81,  82,  83,  85,  87,  89,  90,  92,  93,  94,  95,  98,  99, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 114, 115, 116, 117, 118, 119])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000026B36C6E550>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.

In [3]:
from ultralytics import YOLO

model = YOLO("runs/detect/runs/plate_train/weights/best.pt")

results = model("car2.jpg", save=True)


image 1/1 c:\Users\User\Documents\License_Plate_AI\License_Read\car2.jpg: 384x512 1 1, 1 5, 1 8, 1 9, 1 A01, 1 A04, 1 BKK, 57.1ms
Speed: 3.4ms preprocess, 57.1ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 512)
Results saved to C:\Users\User\Documents\License_Plate_AI\License_Read\runs\detect\predict8


In [4]:
from ultralytics import YOLO
import cv2

model = YOLO("runs/detect/runs/plate_train/weights/best.pt")

img = cv2.imread("car2.jpg")
results = model(img)[0]

upper_line = []
lower_line = []

h = img.shape[0]

for box in results.boxes:
    x1, y1, x2, y2 = box.xyxy[0]
    cls_id = int(box.cls)
    label = model.names[cls_id]

    x_center = int((x1 + x2) / 2)
    y_center = int((y1 + y2) / 2)

    # แยกบรรทัดบน / ล่าง
    if y_center < h / 2:
        upper_line.append((x_center, label))
    else:
        lower_line.append((x_center, label))

# เรียงซ้ายไปขวา
upper_line = sorted(upper_line, key=lambda x: x[0])
lower_line = sorted(lower_line, key=lambda x: x[0])

plate_number = "".join([c[1] for c in upper_line])
province = "".join([c[1] for c in lower_line])

print("ทะเบียน:", plate_number)
print("จังหวัด:", province)


0: 384x512 1 1, 1 5, 1 8, 1 9, 1 A01, 1 A04, 1 BKK, 47.3ms
Speed: 2.8ms preprocess, 47.3ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 512)
ทะเบียน: 5A01A04198
จังหวัด: BKK
